In [89]:
import pandas as pd
import numpy as np
from pathlib import Path

In [90]:
raw_folder = Path("Data_Raw_01")
clean_folder = Path("Data_Clean_02")
clean_folder.mkdir(parents=True, exist_ok=True)

doctors_raw = pd.read_csv(
    raw_folder / "11_إجمالي_عدد_الأطباء_حسب_الدولة_والسنة.csv"
)

beds_raw = pd.read_csv(
    raw_folder / "13_عدد_أسرة_المستشفيات_لكل_10000_نسمة.csv"
)

nurses_raw = pd.read_csv(
    raw_folder / "14_عدد_الممرضين_والقابلات_لكل_10000_نسمة.csv"
)

datasets = {
    "Doctors": doctors_raw,
    "Hospital Beds": beds_raw,
    "Nurses and Midwives": nurses_raw
}

for name, df in datasets.items():
    print(f"\n{name}")
    print("Rows:", len(df))
    print("Columns:", len(df.columns))
    print("Duplicate rows:", df.duplicated().sum())
    print("Missing values:", df.isna().sum().sum())
    display(df.head(3))



Doctors
Rows: 4293
Columns: 34
Duplicate rows: 0
Missing values: 77274


,IndicatorCode,Indicator,ValueType,ParentLocationCode,ParentLocation,Location type,SpatialDimValueCode,Location,Period type,Period,...,FactValueUoM,FactValueNumericLowPrefix,FactValueNumericLow,FactValueNumericHighPrefix,FactValueNumericHigh,Value,FactValueTranslationID,FactComments,Language,DateModified
0,HWF_0002,Medical doctors (number),numeric,SEAR,South-East Asia,Country,IND,India,Year,2024,...,NaN,NaN,NaN,NaN,NaN,1 386 136,NaN,"NHWA data portal, December 2025 update; https:...",EN,2026-01-22T22:00:00.000Z
1,HWF_0002,Medical doctors (number),numeric,AFR,Africa,Country,UGA,Uganda,Year,2024,...,NaN,NaN,NaN,NaN,NaN,10 322,NaN,"NHWA data portal, December 2025 update; https:...",EN,2026-01-22T22:00:00.000Z
2,HWF_0002,Medical doctors (number),numeric,WPR,Western Pacific,Country,PHL,Philippines,Year,2024,...,NaN,NaN,NaN,NaN,NaN,100 608,NaN,"NHWA data portal, December 2025 update; https:...",EN,2026-01-22T22:00:00.000Z



Hospital Beds
Rows: 2958
Columns: 34
Duplicate rows: 0
Missing values: 53244


,IndicatorCode,Indicator,ValueType,ParentLocationCode,ParentLocation,Location type,SpatialDimValueCode,Location,Period type,Period,...,FactValueUoM,FactValueNumericLowPrefix,FactValueNumericLow,FactValueNumericHighPrefix,FactValueNumericHigh,Value,FactValueTranslationID,FactComments,Language,DateModified
0,WHS6_102,Hospital beds (per 10 000 population),numeric,AMR,Americas,Country,BLZ,Belize,Year,2023,...,NaN,NaN,NaN,NaN,NaN,10.0,NaN,WHO PAHO Core Indicators Portal. Region of the...,EN,2025-09-29T21:00:00.000Z
1,WHS6_102,Hospital beds (per 10 000 population),numeric,AFR,Africa,Country,ERI,Eritrea,Year,2023,...,NaN,NaN,NaN,NaN,NaN,10.2,NaN,Provided by MoH during 2025 SDG 3.8.1 country ...,EN,2025-09-29T21:00:00.000Z
2,WHS6_102,Hospital beds (per 10 000 population),numeric,WPR,Western Pacific,Country,COK,Cook Islands,Year,2023,...,NaN,NaN,NaN,NaN,NaN,106.7,NaN,"National Health Information Bulletin, Ministry...",EN,2025-09-29T21:00:00.000Z



Nurses and Midwives
Rows: 3381
Columns: 12
Duplicate rows: 0
Missing values: 0


,IND_ID,IND_CODE,IND_UUID,IND_PER_CODE,DIM_TIME,DIM_TIME_TYPE,DIM_GEO_CODE_M49,DIM_GEO_CODE_TYPE,DIM_PUBLISH_STATE_CODE,IND_NAME,GEO_NAME_SHORT,RATE_PER_10000_N
0,5C8435FHWF_0006,HWF_0006,5C8435F,HWF_0006,1996,YEAR,554,COUNTRY,PUBLISHED,Density of nurses and midwives,New Zealand,97.34
1,5C8435FHWF_0006,HWF_0006,5C8435F,HWF_0006,1996,YEAR,570,COUNTRY,PUBLISHED,Density of nurses and midwives,Niue,56.52
2,5C8435FHWF_0006,HWF_0006,5C8435F,HWF_0006,1996,YEAR,586,COUNTRY,PUBLISHED,Density of nurses and midwives,Pakistan,3.35


In [91]:

doctors = doctors_raw[
    ["SpatialDimValueCode", "Location", "Period", "FactValueNumeric"]
].copy()

doctors = doctors.rename(columns={
    "SpatialDimValueCode": "ISO3",
    "Location": "Country_Source",
    "Period": "Year",
    "FactValueNumeric": "Doctors_Total"
})

doctors["ISO3"] = (
    doctors["ISO3"].astype("string").str.strip().str.upper()
)

doctors["Year"] = pd.to_numeric(
    doctors["Year"], errors="coerce"
).astype("Int64")

doctors["Doctors_Total"] = pd.to_numeric(
    doctors["Doctors_Total"], errors="coerce"
)

print(doctors.dtypes)
print("Rows:", len(doctors))
print(
    "Negative rates:",
    (doctors["Doctors_Total"] < 0).sum()
    
)
display(doctors.head())


ISO3              string[python]
Country_Source            object
Year                       Int64
Doctors_Total              int64
dtype: object
Rows: 4293
Negative rates: 0


,ISO3,Country_Source,Year,Doctors_Total
0,IND,India,2024,1386136
1,UGA,Uganda,2024,10322
2,PHL,Philippines,2024,100608
3,BRN,Brunei Darussalam,2024,1008
4,BWA,Botswana,2024,1080


In [92]:
nurses = nurses_raw[
    [
        "DIM_GEO_CODE_M49",
        "GEO_NAME_SHORT",
        "DIM_TIME",
        "RATE_PER_10000_N"
    ]
].copy()

nurses = nurses.rename(columns={
    "DIM_GEO_CODE_M49": "M49",
    "GEO_NAME_SHORT": "Country_Source",
    "DIM_TIME": "Year",
    "RATE_PER_10000_N": "Nurses_Midwives_Per_10k"
})

nurses["M49"] = (
    nurses["M49"].astype("string").str.strip()
)

nurses["Year"] = pd.to_numeric(
    nurses["Year"], errors="coerce"
).astype("Int64")

nurses["Nurses_Midwives_Per_10k"] = pd.to_numeric(
    nurses["Nurses_Midwives_Per_10k"], errors="coerce"
)

print("Rows:", len(nurses))
print("Missing values:")
print(nurses.isna().sum())

print(
    "Negative rates:",
    (nurses["Nurses_Midwives_Per_10k"] < 0).sum()
)

display(nurses.head())


Rows: 3381
Missing values:
M49                        0
Country_Source             0
Year                       0
Nurses_Midwives_Per_10k    0
dtype: int64
Negative rates: 0


,M49,Country_Source,Year,Nurses_Midwives_Per_10k
0,554,New Zealand,1996,97.34
1,570,Niue,1996,56.52
2,586,Pakistan,1996,3.35
3,591,Panama,1996,10.84
4,616,Poland,1996,62.55


In [93]:
beds = beds_raw[
    ["SpatialDimValueCode", "Location", "Period", "FactValueNumeric"]
].copy()

beds = beds.rename(columns={
    "SpatialDimValueCode": "ISO3",
    "Location": "Country_Source",
    "Period": "Year",
    "FactValueNumeric": "Hospital_Beds_Per_10k"
})

beds["ISO3"] = (
    beds["ISO3"].astype("string").str.strip().str.upper()
)

beds["Year"] = pd.to_numeric(
    beds["Year"], errors="coerce"
).astype("Int64")

beds["Hospital_Beds_Per_10k"] = pd.to_numeric(
    beds["Hospital_Beds_Per_10k"], errors="coerce"
)

print("Rows:", len(beds))
print("Missing values:")
print(beds.isna().sum())
print("ISO3 + Year duplicated  ",
    beds.duplicated(
        subset=["ISO3", "Year"],
        keep=False
    ).sum())
print(
    "Negative rates:",
    (beds["Hospital_Beds_Per_10k"] < 0).sum()
)

display(beds.head())


Rows: 2958
Missing values:
ISO3                     0
Country_Source           0
Year                     0
Hospital_Beds_Per_10k    0
dtype: int64
ISO3 + Year duplicated   0
Negative rates: 0


,ISO3,Country_Source,Year,Hospital_Beds_Per_10k
0,BLZ,Belize,2023,10.02
1,ERI,Eritrea,2023,10.22
2,COK,Cook Islands,2023,106.70
3,GMB,Gambia,2023,11.10
4,SLV,El Salvador,2023,12.07


In [94]:
doctors.duplicated().sum()

np.int64(0)

In [95]:
doctors.isna().sum()

ISO3              0
Country_Source    0
Year              0
Doctors_Total     0
dtype: int64

In [96]:
nurses.isnull().sum()

M49                        0
Country_Source             0
Year                       0
Nurses_Midwives_Per_10k    0
dtype: int64

In [97]:
nurses.duplicated().sum()

np.int64(0)

In [98]:
beds.duplicated().sum()

np.int64(0)

In [99]:
beds.isna().sum()

ISO3                     0
Country_Source           0
Year                     0
Hospital_Beds_Per_10k    0
dtype: int64

In [100]:
population_data = demographics_raw[
    ["ISO3", "Year", "Population"]
].copy()

population_data["Year"] = pd.to_numeric(
    population_data["Year"], errors="coerce"
).astype("Int64")

population_data["Population"] = pd.to_numeric(
    population_data["Population"], errors="coerce"
)

workforce = doctors.merge(
    population_data,
    on=["ISO3", "Year"],
    how="left",
    validate="one_to_one",
    indicator=True
)
print("Doctors rows before merge:", len(doctors))
print("Rows after merge:", len(workforce))
print("\nPopulation match status:")
print(workforce["_merge"].value_counts())

print("\nMissing population after merge:",
      workforce["Population"].isna().sum())

workforce["Doctors_Per_10k"] = (
    workforce["Doctors_Total"]
    / workforce["Population"]
    * 10000
)

workforce["Doctors_Per_10k"] = (
    workforce["Doctors_Per_10k"].round(2)
)

workforce.loc[
    workforce["Population"].isna()
    | (workforce["Population"] <= 0),
    "Doctors_Per_10k"
] = pd.NA

display(
    workforce[
        [
            "ISO3",
            "Country_Source",
            "Year",
            "Doctors_Total",
            "Population",
            "Doctors_Per_10k",
            "_merge"
        ]
    ].head(10)
)


Doctors rows before merge: 4293
Rows after merge: 4293

Population match status:
_merge
both          4280
left_only       13
right_only       0
Name: count, dtype: int64

Missing population after merge: 13


,ISO3,Country_Source,Year,Doctors_Total,Population,Doctors_Per_10k,_merge
0,IND,India,2024,1386136,1.450936e+09,9.55,both
1,UGA,Uganda,2024,10322,5.001509e+07,2.06,both
2,PHL,Philippines,2024,100608,1.158437e+08,8.68,both
3,BRN,Brunei Darussalam,2024,1008,4.627210e+05,21.78,both
4,BWA,Botswana,2024,1080,2.521139e+06,4.28,both
5,KGZ,Kyrgyzstan,2024,11101,7.221868e+06,15.37,both
6,OMN,Oman,2024,11287,5.281538e+06,21.37,both
7,CAN,Canada,2024,113435,4.126233e+07,27.49,both
8,BDI,Burundi,2024,1169,1.404779e+07,0.83,both
9,TZA,United Republic of Tanzania,2024,12023,6.856016e+07,1.75,both


In [101]:
print(
    "Duplicate ISO3 + Year rows:",
    demographics_raw.duplicated(
        subset=["ISO3", "Year"],
        keep=False
    ).sum()
)

print("\nMissing values:")
print(demographics_raw.isna().sum())

print("\nData types:")
print(demographics_raw.dtypes)

print("\nSample data:")
display(demographics_raw.head())

Duplicate ISO3 + Year rows: 0

Missing values:
ISO3                     0
Country                  0
Year                     0
Population              30
Age_0_14_Pct             0
Age_65_Plus_Pct          0
Urban_Population_Pct     0
dtype: int64

Data types:
ISO3                     object
Country                  object
Year                      int64
Population              float64
Age_0_14_Pct            float64
Age_65_Plus_Pct         float64
Urban_Population_Pct    float64
dtype: object

Sample data:


,ISO3,Country,Year,Population,Age_0_14_Pct,Age_65_Plus_Pct,Urban_Population_Pct
0,AFG,Afghanistan,1960,9035043.0,41.946754,2.823362,8.122551
1,AFG,Afghanistan,1961,9214083.0,42.028588,2.809357,8.389595
2,AFG,Afghanistan,1962,9404406.0,42.112814,2.792835,8.662664
3,AFG,Afghanistan,1963,9604487.0,42.235952,2.775057,8.942709
4,AFG,Afghanistan,1964,9814318.0,42.409539,2.757792,9.231134


In [102]:
print("Rows:", len(doctors_raw))
print("\nMissing values:")
print(doctors[
    ["ISO3", "Country_Source", "Year", "Doctors_Total"]
].isna().sum())
print("\nInvalid year values:",
      pd.to_numeric(doctors["Year"], errors="coerce").isna().sum())

print("\nInvalid or negative doctor counts:",
      (
          pd.to_numeric(doctors["Doctors_Total"], errors="coerce").isna()
          | (pd.to_numeric(doctors["Doctors_Total"], errors="coerce") < 0)
      ).sum())

print("\nDuplicate ISO3 + Year rows:",
      doctors.duplicated(
          subset=["ISO3", "Year"],
          keep=False
      ).sum())


Rows: 4293

Missing values:
ISO3              0
Country_Source    0
Year              0
Doctors_Total     0
dtype: int64

Invalid year values: 0

Invalid or negative doctor counts: 0

Duplicate ISO3 + Year rows: 0


In [104]:
workforce_clean = workforce[
    [
        "ISO3",
        "Country",
        "Year",
        "Doctors_Total",
        "Doctors_Per_10k"
    ]
].copy() if "Country" in workforce.columns else workforce[
    [
        "ISO3",
        "Country_Source",
        "Year",
        "Doctors_Total",
        "Doctors_Per_10k"
    ]
].rename(columns={"Country_Source": "Country"})

workforce_clean["Year"] = workforce_clean["Year"].astype("Int64")

print("Final workforce shape:", workforce_clean.shape)
print("Columns:", workforce_clean.columns.tolist())

display(workforce_clean.head())


Final workforce shape: (4293, 5)
Columns: ['ISO3', 'Country', 'Year', 'Doctors_Total', 'Doctors_Per_10k']


,ISO3,Country,Year,Doctors_Total,Doctors_Per_10k
0,IND,India,2024,1386136,9.55
1,UGA,Uganda,2024,10322,2.06
2,PHL,Philippines,2024,100608,8.68
3,BRN,Brunei Darussalam,2024,1008,21.78
4,BWA,Botswana,2024,1080,4.28


In [105]:
dictionary_path = raw_folder / "country_dictionary_v1.csv"

print("File exists:", dictionary_path.exists())

country_dictionary = pd.read_csv(dictionary_path)

print("Rows:", len(country_dictionary))
print("Columns:", country_dictionary.columns.tolist())

display(country_dictionary.head())


File exists: True
Rows: 196
Columns: ['ISO3', 'M49', 'Country', 'WB_Name', 'WHO_Name', 'GHED_Name', 'Nurses_Source_Name', 'Region', 'Income_Group', 'In_GHED', 'In_WHO_Doctors', 'In_WorldBank', 'In_Nurses_Source']


,ISO3,M49,Country,WB_Name,WHO_Name,GHED_Name,Nurses_Source_Name,Region,Income_Group,In_GHED,In_WHO_Doctors,In_WorldBank,In_Nurses_Source
0,AFG,4,Afghanistan,Afghanistan,Afghanistan,Afghanistan,Afghanistan,EMR,Low,True,True,True,True
1,AGO,24,Angola,Angola,Angola,Angola,Angola,AFR,Lower-middle,True,True,True,True
2,ALB,8,Albania,Albania,Albania,Albania,Albania,EUR,Upper-middle,True,True,True,True
3,AND,20,Andorra,Andorra,Andorra,Andorra,Andorra,EUR,High,True,True,True,True
4,ARE,784,United Arab Emirates,United Arab Emirates,United Arab Emirates,United Arab Emirates,United Arab Emirates,EMR,High,True,True,True,True


In [106]:
country_mapping = country_dictionary[["M49", "ISO3", "Country"]].copy()

country_mapping["M49"] = pd.to_numeric(
    country_mapping["M49"], errors="coerce"
).astype("Int64")

country_mapping["ISO3"] = (
    country_mapping["ISO3"].astype("string").str.strip()
)

print("Total mapping rows:", len(country_mapping))
print("Missing M49:", country_mapping["M49"].isna().sum())
print("Missing ISO3:", country_mapping["ISO3"].isna().sum())
print(
    "Duplicate M49 codes:",
    country_mapping.duplicated(subset=["M49"], keep=False).sum()
)
display(country_mapping.head())

Total mapping rows: 196
Missing M49: 0
Missing ISO3: 0
Duplicate M49 codes: 0


,M49,ISO3,Country
0,4,AFG,Afghanistan
1,24,AGO,Angola
2,8,ALB,Albania
3,20,AND,Andorra
4,784,ARE,United Arab Emirates


In [107]:
nurses_prepared["M49"] = pd.to_numeric(
    nurses_prepared["M49"], errors="coerce"
).astype("Int64")

country_mapping["M49"] = pd.to_numeric(
    country_mapping["M49"], errors="coerce"
).astype("Int64")

print("Nurses M49 type:", nurses_prepared["M49"].dtype)
print("Dictionary M49 type:", country_mapping["M49"].dtype)

Nurses M49 type: Int64
Dictionary M49 type: Int64


In [108]:
nurses_mapped = nurses_prepared.merge(
    country_mapping,
    on="M49",
    how="left",
    validate="many_to_one",
    indicator=True
)

print("Total nurses rows:", len(nurses_mapped))
print("Matched rows:", (nurses_mapped["_merge"] == "both").sum())
print("Unmatched rows:", (nurses_mapped["_merge"] == "left_only").sum())

unmatched_m49 = nurses_mapped.loc[
    nurses_mapped["_merge"] == "left_only",
    ["M49", "Country_Source"]
].drop_duplicates()

display(unmatched_m49)
print("Unmatched source rows:",
      (nurses_mapped["_merge"] == "left_only").sum())

print("Unmatched geographic codes:",
      len(unmatched_summary))

Total nurses rows: 3381
Matched rows: 3374
Unmatched rows: 7


,M49,Country_Source
2044,953,Africa
2045,954,Americas
2046,955,South-East Asia
2047,956,Europe
2048,957,Eastern Mediterranean
2049,958,Western Pacific
2718,1,World


Unmatched source rows: 7
Unmatched geographic codes: 7


In [109]:
nurses_countries = nurses_mapped.loc[
    nurses_mapped["_merge"] == "both"
].copy()

nurses_clean = nurses_countries[
    ["ISO3", "Country", "Year", "Nurses_Midwives_Per_10k"]
].copy()

print("Rows before excluding aggregates:", len(nurses_mapped))
print("Rows after excluding aggregates:", len(nurses_clean))
print("Missing ISO3:", nurses_clean["ISO3"].isna().sum())
print("Duplicate ISO3-Year:",
      nurses_clean.duplicated(
          subset=["ISO3", "Year"]
      ).sum())

display(nurses_clean.head())

Rows before excluding aggregates: 3381
Rows after excluding aggregates: 3374
Missing ISO3: 0
Duplicate ISO3-Year: 0


,ISO3,Country,Year,Nurses_Midwives_Per_10k
0,NZL,New Zealand,1996,97.34
1,NIU,Niue,1996,56.52
2,PAK,Pakistan,1996,3.35
3,PAN,Panama,1996,10.84
4,POL,Poland,1996,62.55


In [110]:
print("Total rows:", len(nurses_clean))
print("Missing values:")
print(nurses_clean.isna().sum())
print(
    "Duplicate ISO3-Year:",
    nurses_clean.duplicated(
        subset=["ISO3", "Year"]
    ).sum()
)

print(
    "Negative rates:",
    (nurses_clean["Nurses_Midwives_Per_10k"] < 0).sum()
)

print(
    "Year range:",
    nurses_clean["Year"].min(),
    "-",
    nurses_clean["Year"].max()
)

Total rows: 3374
Missing values:
ISO3                       0
Country                    0
Year                       0
Nurses_Midwives_Per_10k    0
dtype: int64
Duplicate ISO3-Year: 0
Negative rates: 0
Year range: 1990 - 2023


In [118]:
print("Doctors table columns:")
print(workforce_clean.columns.tolist())

print("\nNurses table columns:")
print(nurses_clean.columns.tolist())

print("\nMerged table columns:")
print(workforce_combined.columns.tolist())

Doctors table columns:
['ISO3', 'Country', 'Year', 'Doctors_Total', 'Doctors_Per_10k']

Nurses table columns:
['ISO3', 'Country', 'Year', 'Nurses_Midwives_Per_10k']

Merged table columns:
['ISO3', 'Country_Doctors', 'Year', 'Doctors_Total', 'Doctors_Per_10k', 'Country_Nurses', 'Nurses_Midwives_Per_10k', '_merge']


In [119]:
# Create one country column
workforce_combined["Country"] = (
    workforce_combined["Country_Doctors"]
    .combine_first(workforce_combined["Country_Nurses"])
)

# Remove duplicate country-name columns
workforce_combined = workforce_combined.drop(
    columns=["Country_Doctors", "Country_Nurses"]
)

# Check the result
print("Columns:", workforce_combined.columns.tolist())

print("\nTotal rows:", len(workforce_combined))

print("\nMerge status:")
print(workforce_combined["_merge"].value_counts())

display(workforce_combined.head())

Columns: ['ISO3', 'Year', 'Doctors_Total', 'Doctors_Per_10k', 'Nurses_Midwives_Per_10k', '_merge', 'Country']

Total rows: 4624

Merge status:
_merge
both          3043
left_only     1250
right_only     331
Name: count, dtype: int64


,ISO3,Year,Doctors_Total,Doctors_Per_10k,Nurses_Midwives_Per_10k,_merge,Country
0,AFG,2001,4104.0,2.02,NaN,left_only,Afghanistan
1,AFG,2005,NaN,NaN,6.12,right_only,Afghanistan
2,AFG,2006,4220.0,1.66,4.58,both,Afghanistan
3,AFG,2007,4723.0,1.82,5.18,both,Afghanistan
4,AFG,2008,4834.0,1.83,5.20,both,Afghanistan


In [120]:
# Select the required final columns
workforce_final = workforce_combined[
    [
        "ISO3",
        "Country",
        "Year",
        "Doctors_Total",
        "Doctors_Per_10k",
        "Nurses_Midwives_Per_10k"
    ]
].copy()

# Ensure Year is stored as an integer type
workforce_final["Year"] = workforce_final["Year"].astype("Int64")

# Review the final structure
print("Final rows:", len(workforce_final))
print("Final columns:", workforce_final.columns.tolist())
print("Duplicate ISO3-Year:",
      workforce_final.duplicated(
          subset=["ISO3", "Year"]
      ).sum())

display(workforce_final.head())

Final rows: 4624
Final columns: ['ISO3', 'Country', 'Year', 'Doctors_Total', 'Doctors_Per_10k', 'Nurses_Midwives_Per_10k']
Duplicate ISO3-Year: 0


,ISO3,Country,Year,Doctors_Total,Doctors_Per_10k,Nurses_Midwives_Per_10k
0,AFG,Afghanistan,2001,4104.0,2.02,NaN
1,AFG,Afghanistan,2005,NaN,NaN,6.12
2,AFG,Afghanistan,2006,4220.0,1.66,4.58
3,AFG,Afghanistan,2007,4723.0,1.82,5.18
4,AFG,Afghanistan,2008,4834.0,1.83,5.20


In [121]:
from pathlib import Path

clean_folder = Path("Data_Clean_02")
clean_folder.mkdir(parents=True, exist_ok=True)

workforce_final.to_csv(
    clean_folder / "healthcare_workforce_clean.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Saved to:", clean_folder / "healthcare_workforce_clean.csv")
print("File exists:",
      (clean_folder / "healthcare_workforce_clean.csv").exists())

Saved to: Data_Clean_02\healthcare_workforce_clean.csv
File exists: True


In [122]:
from pathlib import Path

clean_folder = Path("Data_Clean_02")
clean_folder.mkdir(parents=True, exist_ok=True)

capacity_final = beds[
    ["ISO3", "Country_Source", "Year", "Hospital_Beds_Per_10k"]
].copy()

capacity_final.to_csv(
    clean_folder / "healthcare_capacity_clean.csv",
    index=False,
    encoding="utf-8-sig"
)

print("File saved:", (clean_folder / "healthcare_capacity_clean.csv").exists())
print("Rows saved:", len(capacity_final))

File saved: True
Rows saved: 2958
